In [ ]:
# Setup
from pathlib import Path
import sys, json
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data import load_data
from model import split_data, load_model, build_models
from utils import save_fig, plot_confusion

ARTEFACTS_DIR = PROJECT_ROOT / 'artefacts'
with open(ARTEFACTS_DIR / 'model_results.json') as f:
    results = json.load(f)
print(f"Best model: {results['best_model']}")

In [ ]:
# 1. Reload model + test data
best_name = results['best_model']
models = build_models()
df = load_data()
X_train, X_test, y_train, y_test = split_data(df)

# Re-fit the best pipeline so we have a fresh in-memory copy
pipe = models[best_name]
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]

In [ ]:
# 2. Confusion matrix
cm = results['test_results'][best_name]['confusion_matrix']
plot_confusion(cm)
save_fig('04_confusion_matrix.png')
plt.show()

In [ ]:
# 3. Per-model comparison table
rows = []
for name in results['cv_results']:
    cv = results['cv_results'][name]
    tr = results['test_results'][name]
    rows.append({
        'model': name,
        'CV_F1': f"{cv['f1']['mean']:.3f} ± {cv['f1']['std']:.3f}",
        'CV_ROC-AUC': f"{cv['roc_auc']['mean']:.3f} ± {cv['roc_auc']['std']:.3f}",
        'Test_F1': f"{tr['f1']:.3f}",
        'Test_Precision': f"{tr['precision']:.3f}",
        'Test_Recall': f"{tr['recall']:.3f}",
        'Test_ROC-AUC': f"{tr['roc_auc']:.3f}",
    })
comparison = pd.DataFrame(rows).set_index('model')
print(comparison.to_string())
comparison.to_csv(ARTEFACTS_DIR / 'model_comparison.csv')

In [ ]:
# 4. Feature importance
# After preprocessing the column names are no longer the originals — we extract them
# from the fitted ColumnTransformer.
try:
    feature_names = pipe.named_steps['pre'].get_feature_names_out()
    coefs_or_imps = None
    clf = pipe.named_steps['clf']
    if hasattr(clf, 'feature_importances_'):
        coefs_or_imps = clf.feature_importances_
    elif hasattr(clf, 'coef_'):
        coefs_or_imps = clf.coef_[0]
    if coefs_or_imps is not None:
        fi = (pd.DataFrame({'feature': feature_names, 'importance': coefs_or_imps})
                .sort_values('importance', ascending=False)
                .head(15))
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.barplot(data=fi, y='feature', x='importance', palette='viridis', ax=ax)
        ax.set_title(f'Top-15 feature importances — {best_name}')
        save_fig('05_feature_importance.png')
        plt.show()
    else:
        print("This classifier does not expose feature_importances_ or coef_.")
except Exception as e:
    print(f"Could not extract feature importances: {e}")

In [ ]:
# 5. Business-readable summary
summary_lines = [
    "=== BUSINESS SUMMARY ===",
    "Business question: which customers are most likely to churn in the next 30-60 days,",
    "and what drives their decision?",
    "",
    f"Best model: {best_name}",
    f"Test F1:    {results['test_results'][best_name]['f1']:.3f}",
    f"Test ROC-AUC: {results['test_results'][best_name]['roc_auc']:.3f}",
    "",
    "Top churn drivers (from feature importance):",
]
try:
    summary_lines += fi.head(5)['feature'].tolist()
except Exception:
    summary_lines.append("N/A")

summary_lines += [
    "",
    "Recommended retention actions:",
    "- Offer 12-month contract upgrades to month-to-month customers with > 50 monthly charges.",
    "- Proactively reach out to fibre-internet customers in their first 6 months of tenure.",
    "- Consider a loyalty discount for customers paying by electronic check (highest churn rate).",
]
print("\n".join(summary_lines))